## 2.1 Zero-Shot MMLU baseline

In [ ]:
from typing import Any
import re

def parse_mmlu_response(
    response: str,
    mmlu_example: dict[str, Any] | None = None,
):
    chars = re.findall(r'the correct answer is (.)', response.lower())
    for char in chars[::-1]:
        if char in ["a", "b", "c", "d"]:
            return char.upper()
    return None


In [ ]:
from typing import Any
import re

def parse_gsm8k_response(
    response: str,
):
    numbers = re.findall(r'\d+', response)
    if numbers:
        return numbers[-1]
    return None


In [ ]:
from vllm import LLM, SamplingParams
llm = LLM(model="meta-llama/Llama-3.1-8B")

# Create a sampling params object, stopping generation on newline.
sampling_params = SamplingParams(
    temperature=0.0, top_p=1.0, max_tokens=1024, stop=["# Query:"]
)


In [ ]:
from pathlib import Path
mmlu_eval_dir = Path("/home/azureuser/localfiles/cs336-assignment5-alignment-mine/data/mmlu/val")

import pandas as pd
mmlu_examples = []
for file in mmlu_eval_dir.glob("*.csv"):
    subject = file.name.split("_val.csv")[0]
    df = pd.read_csv(file, names=["question", "A", "B", "C", "D", "answer"])
    df["subject"] = subject
    df["options"] = df[["A", "B", "C", "D"]].values.tolist()
    mmlu_examples.extend(df.to_dict("records"))



In [ ]:
gsm8k_file = "/home/azureuser/localfiles/cs336-assignment5-alignment-mine/data/gsm8k/test.jsonl"
df = pd.read_json(gsm8k_file, lines=True)
df["ground_truth"] = df.answer.apply(lambda x: parse_gsm8k_response(x))
gsm8k_examples = df.to_dict(orient="records")

gsm8k_prompt_file = "/home/azureuser/localfiles/cs336-assignment5-alignment-mine/cs336_alignment/prompts/question_only.prompt"
with open(gsm8k_prompt_file) as f:
    gsm8k_prompt_template = f.read()

gsm8k_instructions = [
    gsm8k_prompt_template.format(
        question = example["question"],
    ) for example in gsm8k_examples
]
print(gsm8k_instructions[10])

In [ ]:
import pandas as pd
alpaca_file = "/home/azureuser/localfiles/cs336-assignment5-alignment-mine/data/alpaca_eval/alpaca_eval.jsonl"
df = pd.read_json(alpaca_file, lines=True)
alpaca_examples = df.to_dict(orient="records")


In [ ]:
import pandas as pd
sst_file = "/home/azureuser/localfiles/cs336-assignment5-alignment-mine/data/simple_safety_tests/simple_safety_tests.csv"
df = pd.read_csv(sst_file)


In [ ]:
df.head()

In [ ]:
zero_shot_prompt_file = "/home/azureuser/localfiles/cs336-assignment5-alignment-mine/cs336_alignment/prompts/zero_shot_system_prompt.prompt"

with open(zero_shot_prompt_file) as f:
    zero_shot_prompt_template = f.read()

prompts = [zero_shot_prompt_template.format(instruction=instruction) for instruction in df.prompts_final.tolist()]

In [ ]:
outputs = llm.generate(prompts, sampling_params, use_tqdm=False)
outputs = [opt.outputs[0].text for opt in outputs]

In [ ]:
outputs[0]

In [ ]:
answers = [parse_gsm8k_response(opt) for opt in outputs]
ground_truths = [eg["ground_truth"] for eg in gsm8k_examples]

In [ ]:
sum([ans == gt for ans, gt in zip(answers, ground_truths)]) / len(answers)

## 3 Instruction Fine-tuning
### 3.1 looking at instruction tuning data

In [ ]:
# read training dataset and create a generator that randomly return one data point at a time without replacement

import gzip
import json
import random

file = "/home/azureuser/localfiles/cs336-assignment5-alignment-mine/data/train.jsonl.gz"

def load_jsonl_gz(filepath):
    data = []
    with gzip.open(filepath, 'rt', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

def random_data_generator(data):
    """Generator that yields data points randomly without replacement."""
    indices = list(range(len(data)))
    random.shuffle(indices)
    for idx in indices:
        yield idx, data[idx]

# Load the data
train_data = load_jsonl_gz(file)
print(f"Loaded {len(train_data)} examples")

# Create the generator
data_gen = random_data_generator(train_data)


In [ ]:
# Example usage: get next random sample
idx, sample = next(data_gen)
# print(json.dumps(sample, indent=2))
print(f"{idx}")
print("### Prompt:")
print(sample["prompt"])
print()
print("+"*100)
print()
print("### Respsonse:")
print(sample["response"])

| id | type | quality |
|----|------|---------|
| 156526 | paraphraze | good |
| 22529  |  | incomplete prompt |
| 235  |  | incomplete prompt |
| 4916  | QA | bad response |
| 209725  | QA | good |
| 150636  | story writing | bad response |
| 83707  | story writing | good |
| 139208  | bus routes suggestion | good |
| 42986 | recipe creation | good |
| 164589 | QA | redundant prompt, questionable response |

#### 3.2.1 Data Loader

In [ ]:
alpaca_prompt_file = "/home/azureuser/localfiles/cs336-assignment5-alignment-mine/cs336_alignment/prompts/alpaca_sft.prompt"

with open(alpaca_prompt_file) as f:
    alpaca_sft_template = f.read()

# prompts = [alpaca_sft_template.format(instruction=s["prompt"], response=s["response"]) for s in train_data]

In [ ]:
# file = "/home/azureuser/localfiles/cs336-assignment5-alignment-mine/data/train.jsonl.gz"
def load_jsonl_gz(filepath):
    data = []
    with gzip.open(filepath, 'rt', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

from torch.utils.data import Dataset, DataLoader
import random
from transformers import PreTrainedTokenizerBase
import torch

alpaca_prompt_file = "/home/azureuser/localfiles/cs336-assignment5-alignment-mine/cs336_alignment/prompts/alpaca_sft.prompt"
with open(alpaca_prompt_file, encoding="utf-8") as f:
    ALPACA_SFT_TEMPLATE = f.read()

from rlhf_helper_methods import SFTDataset


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

file_path = "/home/azureuser/localfiles/cs336-assignment5-alignment-mine/data/train.jsonl.gz"
file_path = "/home/azureuser/localfiles/cs336-assignment5-alignment-mine/tests/fixtures/sft_sample.jsonl"
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B")

ds = SFTDataset(
    tokenizer,
    file_path,
    seq_length=32,
)

In [ ]:
def iterate_batches(
    dataset: Dataset,
    batch_size: int,
    shuffle: bool,
):
    loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle
    )
    return loader


In [ ]:
loader = iterate_batches(ds, 2, True)
gen = iter(loader)
# next(gen)